In [14]:
from langchain_community.embeddings import DashScopeEmbeddings
from pymilvus import MilvusClient, AnnSearchRequest, WeightedRanker, RRFRanker

from app.core.config import settings

# 1.准备客户端
client = MilvusClient(uri=settings.rag.milvus_url)

# 2.准备向量模型
embedding = DashScopeEmbeddings(
    model="qwen3.7-text-embedding"
)

# 3.准备集合名称
collection_name = "collection_name_1"

# 4.定义混合检索的函数
def hybrid_search(query: str, ranker, filter_conditions: str):
    # 4.1 稠密向量检索请求
    dense_search = AnnSearchRequest(
        data=[embedding.embed_query(query)],
        anns_field="dense",
        param={"metric_type": "COSINE"},
        limit=3,
        filter=filter_conditions
    )
    # 4.2 稀疏向量检索请求
    sparse_search = AnnSearchRequest(
        data=[query],
        anns_field="sparse",
        param={"metric_type": "BM25"},
        limit=3,
        filter=filter_conditions
    )
    # 4.3 组装请求列表
    request = [dense_search, sparse_search]
    # 4.4 执行正式的请求
    search_result = client.hybrid_search(   # List[List[dict]]
        collection_name, request,
        ranker, limit=3,
        output_fields=["h2", "content"]
    )
    # 4.5 测试查询结果
    for hits in search_result:
        for hit in hits:
            print("======================")
            print(hit)

In [ ]:
# 加权融合排序
ranker = WeightedRanker(0.6, 0.4)
# hybrid_search("教育的负面作用是什么?", ranker)
hybrid_search("给教师的建议是谁写的?", ranker)

In [ ]:
# RRF融合排序
ranker = RRFRanker(k=60)
# hybrid_search("教育的负面作用是什么?", ranker)
hybrid_search("给教师的建议是谁写的?", ranker)

In [6]:
# 自定义重排序

# cross-encoder rerank
from pymilvus import Function, FunctionType


def create_cross_encoder_ranker(queries: list[str]):
    return Function(
        name="dashscope_semantic_ranker",      # ranker名词，唯一即可
        input_field_names=["content"],         # 指定原始文档字段
        function_type=FunctionType.RERANK,     # ranker类型，这里是固定值
        params={
            "reranker": "model",           # rerank类型，基于模型rerank，也就是cross-encoder
            "provider": "ali",             # rerank模型提供者
            "model_name": "qwen3-vl-rerank", # rerank模型名称
            "queries": queries,            # 查询条件
            "max_client_batch_size": 5,    # 单批次中处理的文档最大数量
        }
    )

In [ ]:
query = "给教师的建议是谁写的"
# 1.创建一个交叉编码ranker
ranker = create_cross_encoder_ranker([query])
# 2.混合检索
hybrid_search(query, ranker, filter_conditions="content like '%教育目的%'")